# Iran War and Inflation: a FRED notebook

In this lecture notebook we use simple FRED csv links to study how an Iran war shock could push inflation through the energy channel.  
We will look at three things:
- daily oil prices
- weekly gasoline prices
- slower CPI data and faster market-based inflation expectations

The goal is not to prove that one event explains every price movement. The goal is to show the main transmission channel and why high-frequency inflation proxies can be useful when CPI arrives with a lag.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

plt.style.use('seaborn-v0_8-whitegrid')

## Part 1. Load the data from FRED with direct csv links

This follows the minimal URL style exactly: `fredgraph.csv?id=SERIES&cosd=START&coed=END`.  
No API key is needed for this version.

In [ ]:
oil = pd.read_csv('https://fred.stlouisfed.org/graph/fredgraph.csv?id=DCOILWTICO&cosd=2021-01-01&coed=2026-05-26')
gas = pd.read_csv('https://fred.stlouisfed.org/graph/fredgraph.csv?id=GASREGW&cosd=2021-01-01&coed=2026-05-26')
cpi = pd.read_csv('https://fred.stlouisfed.org/graph/fredgraph.csv?id=CPIAUCSL&cosd=2021-01-01&coed=2026-05-26')
breakeven = pd.read_csv('https://fred.stlouisfed.org/graph/fredgraph.csv?id=T5YIE&cosd=2021-01-01&coed=2026-05-26')

In [ ]:
oil['DATE'] = pd.to_datetime(oil['DATE'])
oil['DCOILWTICO'] = pd.to_numeric(oil['DCOILWTICO'], errors='coerce')
oil = oil.dropna()

gas['DATE'] = pd.to_datetime(gas['DATE'])
gas['GASREGW'] = pd.to_numeric(gas['GASREGW'], errors='coerce')
gas = gas.dropna()

cpi['DATE'] = pd.to_datetime(cpi['DATE'])
cpi['CPIAUCSL'] = pd.to_numeric(cpi['CPIAUCSL'], errors='coerce')
cpi = cpi.dropna()

breakeven['DATE'] = pd.to_datetime(breakeven['DATE'])
breakeven['T5YIE'] = pd.to_numeric(breakeven['T5YIE'], errors='coerce')
breakeven = breakeven.dropna()

In [ ]:
oil.head()

## Part 2. The fast energy channel

If an Iran war raises inflation pressure, the cleanest first place to look is energy. Oil prices can move every day, and gasoline prices can move every week.

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(12, 8), sharex=True)

axes[0].plot(oil['DATE'], oil['DCOILWTICO'], color='firebrick')
axes[0].set_title('WTI crude oil price')
axes[0].set_ylabel('Dollars per barrel')

axes[1].plot(gas['DATE'], gas['GASREGW'], color='darkorange')
axes[1].set_title('US regular gasoline price')
axes[1].set_ylabel('Dollars per gallon')
axes[1].set_xlabel('Date')

plt.tight_layout()
plt.show()

**What this graph shows:**  
Oil and gasoline react much faster than CPI. If conflict around Iran disrupts energy supply or raises shipping risk, this is where we should expect to see the first inflation signal. That is why these series are useful for a real-time classroom discussion even before the monthly CPI release arrives.

In [ ]:
oil_monthly = oil.set_index('DATE').resample('MS').mean().reset_index()
gas_monthly = gas.set_index('DATE').resample('MS').mean().reset_index()

cpi['cpi_yoy'] = cpi['CPIAUCSL'].pct_change(12) * 100

inflation_compare = cpi.merge(oil_monthly, on='DATE', how='left')
inflation_compare = inflation_compare.merge(gas_monthly, on='DATE', how='left')
inflation_compare.tail()

In [ ]:
fig, ax1 = plt.subplots(figsize=(12, 5))

ax1.plot(inflation_compare['DATE'], inflation_compare['cpi_yoy'], color='navy', linewidth=2)
ax1.set_ylabel('CPI inflation, % year over year', color='navy')
ax1.set_xlabel('Date')
ax1.set_title('CPI responds more slowly than energy prices')

ax2 = ax1.twinx()
ax2.plot(inflation_compare['DATE'], inflation_compare['DCOILWTICO'], color='firebrick', alpha=0.6)
ax2.set_ylabel('WTI oil price', color='firebrick')

plt.tight_layout()
plt.show()

**What this graph shows:**  
CPI is the official inflation benchmark, but it moves slowly because it is monthly and usually discussed with a publication lag. Oil can jump immediately, while CPI tends to reflect that pressure later as higher energy costs flow into transportation, utilities, and other prices.

In [ ]:
breakeven_monthly = breakeven.set_index('DATE').resample('MS').mean().reset_index()
expectations = cpi[['DATE', 'cpi_yoy']].merge(breakeven_monthly, on='DATE', how='inner')
expectations.tail()

In [ ]:
fig, ax = plt.subplots(figsize=(12, 5))

ax.plot(expectations['DATE'], expectations['cpi_yoy'], label='CPI inflation, yoy', color='navy', linewidth=2)
ax.plot(expectations['DATE'], expectations['T5YIE'], label='5-year breakeven inflation', color='seagreen', linewidth=2)
ax.set_title('A faster alternative to CPI: market inflation expectations')
ax.set_ylabel('Percent')
ax.set_xlabel('Date')
ax.legend()

plt.tight_layout()
plt.show()

**What this graph shows:**  
The breakeven inflation rate is a market-based measure that updates much faster than CPI. If oil rises during an Iran war and breakevens rise too, markets are telling us that traders expect broader inflation pressure, not just a temporary one-week gasoline spike.

## Final takeaway

- A conflict involving Iran would most likely show up first through oil and gasoline.  
- CPI is still important, but it is too slow to be the only thing we watch in real time.  
- Market-based inflation expectations such as `T5YIE` are a useful alternative because they update quickly.  
- So the inflation story here is really an **energy channel story first, CPI confirmation story second**.